<a href="https://colab.research.google.com/github/bakhita11/3D-CNN-Transformer-Fusion-for-Spanish-Sign-Language-Recognition-/blob/main/mosfet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================

!pip install scikit-optimize

# ==============================
# IMPORT LIBRARIES
# ==============================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Metrics for proper evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from skopt import gp_minimize
from skopt.space import Real

# ==============================
# SET RANDOM SEED
# ==============================

np.random.seed(42)
torch.manual_seed(42)

# ==============================
# DATASET GENERATION
# ==============================

def generate_dataset(num_samples=30000):

    # Generate transistor parameters
    L = np.random.uniform(2e-9, 100e-9, num_samples)
    W = np.random.uniform(0.2e-6, 10e-6, num_samples)
    VDD = np.random.uniform(0.5, 1.2, num_samples)
    Temp = np.random.uniform(250, 400, num_samples)
    Vth = np.random.normal(0.4, 0.05, num_samples)

    time = np.random.uniform(0, 1e-6, num_samples)

    # Physics-inspired behavior
    current = (W / L) * np.maximum((VDD - Vth), 0)**2
    delay = L / (current + 1e-9)

    # Smooth switching curve
    Vout = VDD / (1 + np.exp(-12 * (time - delay)))

    # Add noise
    noise = np.random.normal(0, 0.015, num_samples)
    Vout = Vout + noise

    data = pd.DataFrame({
        'L': L,
        'W': W,
        'VDD': VDD,
        'Temp': Temp,
        'Vth': Vth,
        'time': time,
        'Vout': Vout
    })

    return data

data = generate_dataset()

# ==============================
# FEATURE ENGINEERING (VERY IMPORTANT)
# ==============================

# These features improve learning significantly
data['W_over_L'] = data['W'] / data['L']
data['VDD_minus_Vth'] = data['VDD'] - data['Vth']

# ==============================
# DATA PREPROCESSING
# ==============================

X = data[['L', 'W', 'VDD', 'Temp', 'Vth', 'time', 'W_over_L', 'VDD_minus_Vth']]
y = data[['Vout']]

# Normalize inputs
X_scaler = MinMaxScaler()
X_scaled = X_scaler.fit_transform(X)

# Normalize output (CRITICAL FIX)
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y)

# Split data
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.125, random_state=42
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# ==============================
# IMPROVED MODEL
# ==============================

class TransistorModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(8, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = TransistorModel()

criterion = nn.MSELoss()

# Lower learning rate for better convergence
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Scheduler improves training stability
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

# ==============================
# TRAINING LOOP
# ==============================

epochs = 150

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()

    train_out = model(X_train)
    train_loss = criterion(train_out, y_train)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    train_pred = train_out.detach().numpy()
    train_true = y_train.numpy()

    train_r2 = r2_score(train_true, train_pred)

    # ---- VALIDATION ----
    model.eval()
    with torch.no_grad():

        val_out = model(X_val)
        val_loss = criterion(val_out, y_val)

        val_pred = val_out.numpy()
        val_true = y_val.numpy()

        val_r2 = r2_score(val_true, val_pred)

    scheduler.step(val_loss)

    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"\nEpoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {train_loss.item():.6f} | Train R2: {train_r2:.4f}")
        print(f"Val   Loss: {val_loss.item():.6f} | Val   R2: {val_r2:.4f}")

# ==============================
# FINAL TEST EVALUATION
# ==============================

model.eval()
with torch.no_grad():

    test_out = model(X_test)

    # Convert back to real scale
    test_pred = y_scaler.inverse_transform(test_out.numpy())
    test_true = y_scaler.inverse_transform(y_test.numpy())

    test_loss = mean_squared_error(test_true, test_pred)
    test_r2 = r2_score(test_true, test_pred)
    mae = mean_absolute_error(test_true, test_pred)
    rmse = np.sqrt(test_loss)

print("\n===== FINAL TEST RESULTS =====")
print(f"Test R2: {test_r2:.4f}")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")

# ==============================
# BAYESIAN OPTIMIZATION
# ==============================

def objective(params):

    L, W = params

    VDD = 1.0
    Temp = 300
    Vth = 0.4
    time = 0.5e-6

    # Feature engineering must match training
    W_over_L = W / L
    VDD_minus_Vth = VDD - Vth

    input_data = np.array([[L, W, VDD, Temp, Vth, time, W_over_L, VDD_minus_Vth]])
    input_scaled = X_scaler.transform(input_data)

    input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

    with torch.no_grad():
        pred_scaled = model(input_tensor).numpy()
        pred_real = y_scaler.inverse_transform(pred_scaled)[0][0]

    return abs(1.0 - pred_real)

space = [
    Real(2e-9, 100e-9),
    Real(0.2e-6, 10e-6)
]

result = gp_minimize(objective, space, n_calls=20, random_state=42)

print("\nOptimal Parameters Found:")
print(f"L = {result.x[0]:.2e} m")
print(f"W = {result.x[1]:.2e} m")

/tmp/ipykernel_11191/3681090739.py:53: RuntimeWarning: overflow encountered in exp
  Vout = VDD / (1 + np.exp(-12 * (time - delay)))



Epoch [10/150]
Train Loss: 0.038952 | Train R2: -0.6175
Val   Loss: 0.310709 | Val   R2: -11.7042

Epoch [20/150]
Train Loss: 0.042476 | Train R2: -0.7638
Val   Loss: 0.097286 | Val   R2: -2.9778

Epoch [30/150]
Train Loss: 0.017564 | Train R2: 0.2706
Val   Loss: 0.051873 | Val   R2: -1.1210

Epoch [40/150]
Train Loss: 0.014889 | Train R2: 0.3817
Val   Loss: 0.013840 | Val   R2: 0.4341

Epoch [50/150]
Train Loss: 0.012468 | Train R2: 0.4823
Val   Loss: 0.003403 | Val   R2: 0.8609

Epoch [60/150]
Train Loss: 0.010948 | Train R2: 0.5454
Val   Loss: 0.002936 | Val   R2: 0.8799

Epoch [70/150]
Train Loss: 0.010677 | Train R2: 0.5566
Val   Loss: 0.003556 | Val   R2: 0.8546

Epoch [80/150]
Train Loss: 0.010164 | Train R2: 0.5780
Val   Loss: 0.002780 | Val   R2: 0.8863

Epoch [90/150]
Train Loss: 0.009705 | Train R2: 0.5970
Val   Loss: 0.002466 | Val   R2: 0.8992

Epoch [100/150]
Train Loss: 0.009362 | Train R2: 0.6112
Val   Loss: 0.002444 | Val   R2: 0.9001

Epoch [110/150]
Train Loss: 0.00

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/valida


Optimal Parameters Found:
L = 5.22e-08 m
W = 5.33e-06 m
